# 01. Training Notebook (MPS Safe Setup)

This notebook trains **YOLOv10n** and **YOLOv8n** on a reduced Pascal VOC setup
that is conservative for Apple Silicon memory:

- `device = mps` when available
- small batch size
- reduced image size
- no training plots during fitting
- explicit memory cleanup before each run

The goal is to keep the workflow simple and stable on a Mac Mini.

## What This Notebook Produces

After `Run All`, you should obtain two run folders:

- `results/mps_runs/yolov10n_main_mps_sgd_safe`
- `results/mps_runs/yolov8n_main_mps_sgd_safe`

Each run stores `best.pt`, `last.pt`, and `results.csv`.
The visualization notebook reads these folders directly.

In [ ]:
import gc
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from ultralytics import settings as ul_settings

ROOT = Path(r"/Users/guillaumerabeau/deep-learning-paper-YOLOv10")
os.chdir(ROOT)

RESULTS = ROOT / 'results' / 'mps_runs'
DATASETS_DIR = ROOT / 'data' / 'datasets'
RESULTS.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
ul_settings.update({'datasets_dir': str(DATASETS_DIR), 'runs_dir': str(RESULTS)})

SEED = 42
DATA_YAML = 'VOC.yaml'
IMGSZ = 384
BATCH = 4
EPOCHS = 40
FRACTION = 0.10
LR0 = 0.01
LRF = 0.01
WARMUP_EPOCHS = 3.0
MOMENTUM = 0.937
WEIGHT_DECAY = 5e-4
OPTIMIZER = 'SGD'
WORKERS = 0
RUN_TAG = 'mps_sgd_safe'

MODELS = {
    'YOLOv10n': 'yolov10n.pt',
    'YOLOv8n': 'yolov8n.pt',
}

print({
    'root': str(ROOT),
    'device': DEVICE,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'epochs': EPOCHS,
    'fraction': FRACTION,
})

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available() and hasattr(torch, 'mps') and hasattr(torch.mps, 'manual_seed'):
        torch.mps.manual_seed(seed)

def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def train_if_needed(model_name: str) -> dict:
    run_name = f"{Path(model_name).stem}_main_{RUN_TAG}"
    run_dir = RESULTS / run_name
    best = run_dir / 'weights' / 'best.pt'
    last = run_dir / 'weights' / 'last.pt'
    csv_path = run_dir / 'results.csv'

    if best.exists() and last.exists() and csv_path.exists():
        print(f'[skip] {run_name}')
    else:
        print(f'[train] {run_name} on {DEVICE}')
        set_seed(SEED)
        gc.collect()
        if DEVICE == 'mps' and hasattr(torch, 'mps'):
            torch.mps.empty_cache()
        model = YOLO(model_name)
        model.train(
            data=DATA_YAML,
            epochs=EPOCHS,
            fraction=FRACTION,
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            optimizer=OPTIMIZER,
            lr0=LR0,
            lrf=LRF,
            cos_lr=True,
            warmup_epochs=WARMUP_EPOCHS,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
            workers=WORKERS,
            cache=False,
            amp=False,
            pretrained=True,
            seed=SEED,
            deterministic=True,
            patience=100,
            project=str(RESULTS),
            name=run_name,
            exist_ok=True,
            save=True,
            plots=False,
            verbose=False,
        )

    return {{
        'run_name': run_name,
        'run_dir': str(run_dir),
        'best': str(best),
        'last': str(last),
        'results_csv': str(csv_path),
    }}

In [ ]:
run_info = {name: train_if_needed(weights) for name, weights in MODELS.items()}
save_json(run_info, RESULTS / 'run_info_safe.json')
pd.DataFrame([
    {
        'model': name,
        'run_dir': info['run_dir'],
        'best_exists': Path(info['best']).exists(),
        'last_exists': Path(info['last']).exists(),
    }
    for name, info in run_info.items()
])

## Next Step

Open `notebooks/02_viz_compare_yolov10_mps.ipynb` and run all cells.
That notebook compares the two saved runs directly.